In [0]:
spark.conf.set(
    "fs.azure.account.key.jayeshstorageadlake.dfs.core.windows.net",
    "(access key)"
)



In [0]:
dbutils.fs.ls("abfss://silver@jayeshstorageadlake.dfs.core.windows.net/")

[FileInfo(path='abfss://silver@jayeshstorageadlake.dfs.core.windows.net/Amazon Sale Report.txt', name='Amazon Sale Report.txt', size=74754638, modificationTime=1781091213000),
 FileInfo(path='abfss://silver@jayeshstorageadlake.dfs.core.windows.net/_SUCCESS', name='_SUCCESS', size=0, modificationTime=1781107365000),
 FileInfo(path='abfss://silver@jayeshstorageadlake.dfs.core.windows.net/part-00000-41254873-d681-4779-b83d-56f7d87955a8-c000.csv', name='part-00000-41254873-d681-4779-b83d-56f7d87955a8-c000.csv', size=18578863, modificationTime=1781107365000),
 FileInfo(path='abfss://silver@jayeshstorageadlake.dfs.core.windows.net/part-00001-41254873-d681-4779-b83d-56f7d87955a8-c000.csv', name='part-00001-41254873-d681-4779-b83d-56f7d87955a8-c000.csv', size=18548896, modificationTime=1781107365000),
 FileInfo(path='abfss://silver@jayeshstorageadlake.dfs.core.windows.net/part-00002-41254873-d681-4779-b83d-56f7d87955a8-c000.csv', name='part-00002-41254873-d681-4779-b83d-56f7d87955a8-c000.csv',

In [0]:
df = spark.read.csv(
    "abfss://silver@jayeshstorageadlake.dfs.core.windows.net/",
    header=True,
    inferSchema=True
)

df.count()

128975

In [0]:
df.groupBy("fulfilled-by").count().show()

+------------+-----+
|fulfilled-by|count|
+------------+-----+
|     UNKNOWN|89698|
|   Easy Ship|39277|
+------------+-----+



In [0]:
df.columns

['index',
 'Order ID',
 'Date',
 'Status',
 'Fulfilment',
 'Sales Channel ',
 'ship-service-level',
 'Style',
 'SKU',
 'Category',
 'Size',
 'ASIN',
 'Courier Status',
 'Qty',
 'currency',
 'Amount',
 'ship-city',
 'ship-state',
 'ship-postal-code',
 'ship-country',
 'promotion-ids',
 'B2B',
 'fulfilled-by']

In [0]:
gold_df = spark.read.csv(
    "abfss://silver@jayeshstorageadlake.dfs.core.windows.net/",
    header=True,
    inferSchema=True
)

In [0]:
gold_df.printSchema()

root
 |-- index: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Date: date (nullable = true)
 |-- Status: string (nullable = true)
 |-- Fulfilment: string (nullable = true)
 |-- Sales Channel : string (nullable = true)
 |-- ship-service-level: string (nullable = true)
 |-- Style: string (nullable = true)
 |-- SKU: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Size: string (nullable = true)
 |-- ASIN: string (nullable = true)
 |-- Courier Status: string (nullable = true)
 |-- Qty: integer (nullable = true)
 |-- currency: string (nullable = true)
 |-- Amount: double (nullable = true)
 |-- ship-city: string (nullable = true)
 |-- ship-state: string (nullable = true)
 |-- ship-postal-code: double (nullable = true)
 |-- ship-country: string (nullable = true)
 |-- promotion-ids: string (nullable = true)
 |-- B2B: boolean (nullable = true)
 |-- fulfilled-by: string (nullable = true)



In [0]:
sales_by_category = (
    gold_df
    .groupBy("Category")
    .sum("Amount")
    .withColumnRenamed("sum(Amount)", "Total_Sales")
)

sales_by_category.show()

+-------------+--------------------+
|     Category|         Total_Sales|
+-------------+--------------------+
| Ethnic Dress|           791217.66|
|          Top|   5347792.300000001|
|          Set|       3.920412403E7|
|        Saree|  123933.75999999998|
|       Bottom|  150667.97999999998|
|Western Dress|1.1216072690000003E7|
|       Blouse|  458408.18000000005|
|        kurta| 2.129954670000001E7|
|      Dupatta|               915.0|
+-------------+--------------------+



In [0]:
sales_by_category.write.mode("overwrite") \
.option("header","true") \
.csv("abfss://gold@jayeshstorageadlake.dfs.core.windows.net/sales_by_category")

In [0]:
sales_by_state = (
    gold_df
    .groupBy("ship-state")
    .sum("Amount")
    .withColumnRenamed("sum(Amount)", "Total_Sales")
)

sales_by_state.show()

+--------------------+------------------+
|          ship-state|       Total_Sales|
+--------------------+------------------+
|     DADRA AND NAGAR|          42138.92|
|              SIKKIM|         138125.66|
|               delhi|          16553.62|
|           MEGHALAYA|118949.90000000001|
|                  NL|             961.0|
|              Odisha|          12429.76|
|         WEST BENGAL|3507880.4399999995|
|              Punjab|           22013.0|
|Punjab/Mohali/Zir...|             568.0|
|                 GOA|         619437.85|
|        CHHATTISGARH| 570485.8300000001|
|           RAJASTHAN|         1716802.4|
|             Manipur|            2631.0|
|              punjab|            8622.0|
|             TRIPURA|           92548.4|
|               DELHI| 4235215.970000001|
|                 Goa|           15781.0|
|    HIMACHAL PRADESH|         503364.51|
|               BIHAR|1394388.3199999998|
|          CHANDIGARH|202686.05000000002|
+--------------------+------------

In [0]:
sales_by_state.write.mode("overwrite") \
.option("header","true") \
.csv("abfss://gold@jayeshstorageadlake.dfs.core.windows.net/sales_by_state")

In [0]:
order_status = (
    gold_df
    .groupBy("Status")
    .count()
)

order_status.show()

+--------------------+-----+
|              Status|count|
+--------------------+-----+
|SHIPPED - DELIVER...|28769|
|SHIPPED - LOST IN...|    5|
|SHIPPED - OUT FOR...|   35|
|           CANCELLED|18332|
|SHIPPED - REJECTE...|   11|
| SHIPPED - PICKED UP|  973|
|SHIPPED - RETURNE...| 1953|
|SHIPPED - RETURNI...|  145|
|             SHIPPED|77804|
|             PENDING|  658|
|PENDING - WAITING...|  281|
|            SHIPPING|    8|
|   SHIPPED - DAMAGED|    1|
+--------------------+-----+



In [0]:
order_status.write.mode("overwrite") \
.option("header","true") \
.csv("abfss://gold@jayeshstorageadlake.dfs.core.windows.net/order_status")

In [0]:
fulfillment_summary = (
    gold_df
    .groupBy("fulfilled-by")
    .count()
)

fulfillment_summary.show()

+------------+-----+
|fulfilled-by|count|
+------------+-----+
|     UNKNOWN|89698|
|   Easy Ship|39277|
+------------+-----+



In [0]:
fulfillment_summary.write.mode("overwrite") \
.option("header","true") \
.csv("abfss://gold@jayeshstorageadlake.dfs.core.windows.net/fulfillment_summary")

In [0]:
top_products = (
    gold_df
    .groupBy("SKU")
    .sum("Amount")
    .withColumnRenamed("sum(Amount)", "Total_Sales")
    .orderBy(col("Total_Sales").desc())
)

top_products.show(10, False)

+---------------+------------------+
|SKU            |Total_Sales       |
+---------------+------------------+
|J0230-SKD-M    |527699.2          |
|JNE3797-KR-L   |524581.77         |
|J0230-SKD-S    |479937.14         |
|JNE3797-KR-M   |454290.16000000003|
|JNE3797-KR-S   |407302.57000000007|
|JNE3797-KR-XL  |332155.23999999993|
|J0230-SKD-L    |305616.95         |
|JNE3797-KR-XS  |303616.69999999995|
|SET268-KR-NP-XL|284058.96         |
|JNE3797-KR-XXXL|276375.80000000005|
+---------------+------------------+
only showing top 10 rows


In [0]:
top_products.write.mode("overwrite") \
.option("header","true") \
.csv("abfss://gold@jayeshstorageadlake.dfs.core.windows.net/top_products")

In [0]:
from pyspark.sql.functions import month

monthly_sales = (
    gold_df
    .groupBy(month("Date").alias("Month"))
    .sum("Amount")
)

monthly_sales.show()

+-----+--------------------+
|Month|         sum(Amount)|
+-----+--------------------+
|    4|2.8838708320000004E7|
|    3|  101683.84999999998|
|    5|2.6226476749999918E7|
|    6|2.3425809379999973E7|
+-----+--------------------+



In [0]:
monthly_sales.write.mode("overwrite") \
.option("header","true") \
.csv("abfss://gold@jayeshstorageadlake.dfs.core.windows.net/monthly_sales")